# 📘 Pillow 소개와 기본 조작

**Pillow**는 파이썬에서 가장 널리 쓰이는 이미지 처리 라이브러리입니다.
PIL(Python Imaging Library)의 후속 프로젝트로, 직관적인 API로 이미지를 쉽게 다룰 수 있습니다.

**Pillow vs OpenCV 비교:**

| 특징 | Pillow | OpenCV |
|------|--------|--------|
| 설계 철학 | 이미지 편집/변환 | 컴퓨터 비전/실시간 처리 |
| 데이터 형식 | PIL Image 객체 | NumPy 배열 (BGR) |
| 색상 순서 | RGB | BGR |
| 한글 텍스트 | 자연스러운 렌더링 | 기본 지원 안 함 |
| EXIF/메타데이터 | 풍부한 지원 | 제한적 |
| 필터/효과 | 다양한 내장 필터 | 커널 기반 처리 |
| 적합 분야 | 웹 이미지, 썸네일, 포맷 변환 | 객체 인식, 영상 분석 |

**학습 목표:**
- 이미지 열기, 만들기, 저장하기
- 크기 조정, 자르기, 회전
- 색상 모드 변환 (RGB, L, RGBA 등)
- EXIF 메타데이터 읽기

## 1. 이미지 열기, 만들기, 저장하기

Pillow의 핵심은 `Image` 클래스입니다.
`Image.open()`으로 파일을 열고, `Image.new()`로 새 이미지를 만듭니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  Pillow 임포트와 기본 조작               │
# └─────────────────────────────────────────┘

from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageEnhance
import numpy as np
import matplotlib.pyplot as plt

print(f'Pillow 버전: {Image.__version__}')

# 새 이미지 만들기
img_red = Image.new('RGB', (300, 200), color=(255, 80, 80))
img_white = Image.new('RGB', (300, 200), color='white')
img_alpha = Image.new('RGBA', (300, 200), (0, 150, 255, 128))  # 반투명

# 이미지 속성 확인
print(f'\n=== 이미지 속성 ===')
print(f'크기: {img_red.size}         # (너비, 높이)')
print(f'모드: {img_red.mode}            # RGB, L(흑백), RGBA 등')
print(f'포맷: {img_red.format}          # 파일에서 열 때만 설정됨')

# 이미지 저장
import tempfile, os, shutil
tmpdir = tempfile.mkdtemp()
img_red.save(os.path.join(tmpdir, 'red.png'))
img_alpha.save(os.path.join(tmpdir, 'blue_alpha.png'))

# 다시 열기
img_loaded = Image.open(os.path.join(tmpdir, 'red.png'))
print(f'\n불러온 이미지 크기: {img_loaded.size}')
print(f'불러온 이미지 모드: {img_loaded.mode}')

# 시각화
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(img_red); axes[0].set_title('빨강 (RGB)')
axes[1].imshow(img_white); axes[1].set_title('흰 배경')
axes[2].imshow(img_alpha); axes[2].set_title('반투명 파랑 (RGBA)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

img_loaded.close()  # 파일 핸들 닫기 (Windows 필수)
shutil.rmtree(tmpdir)
print('\nPillow 기본 조작 완료!')

## 2. 크기 조정, 자르기, 회전

이미지의 크기를 바꾸거나 원하는 부분만 잘라내거나 회전할 수 있습니다.

> 💡 Pillow는 **(너비, 높이)** 순서를 사용합니다. NumPy/OpenCV는 (높이, 너비) 순서이므로 주의하세요!

In [ ]:
# ┌─────────────────────────────────────────┐
# │  크기 조정, 자르기, 회전               │
# │  resize, crop, rotate, transpose       │
# └─────────────────────────────────────────┘

# 테스트 이미지 생성
img = Image.new('RGB', (400, 300), 'white')
draw = ImageDraw.Draw(img)
for y in range(300):  # 그라데이션 배경
    r = int(255 * y / 300)
    g = int(100 * (1 - y / 300))
    b = 200
    draw.line([(0, y), (400, y)], fill=(r, g, b))
draw.rectangle([50, 50, 200, 150], fill='orange', outline='black', width=3)
draw.ellipse([220, 50, 370, 200], fill='cyan', outline='navy', width=3)
draw.text((100, 10), 'Pillow Test', fill='white')

# 크기 조정
img_half = img.resize((200, 150))  # 절반 크기
img_thumb = img.copy()
img_thumb.thumbnail((100, 100))  # 비율 유지 축소

# 자르기 crop((left, upper, right, lower))
img_cropped = img.crop((50, 50, 300, 200))

# 회전
img_rotated = img.rotate(30, expand=True)  # 30도, 캔버스 확장

# 뒤집기
img_flip = img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)  # 좌우

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
titles = ['원본 (400x300)', '절반 크기', '썸네일 (100px)',
          '자르기 (crop)', '30도 회전 (expand)', '좌우 뒤집기']
images = [img, img_half, img_thumb, img_cropped, img_rotated, img_flip]
for ax, im, t in zip(axes.flat, images, titles):
    ax.imshow(im)
    ax.set_title(f'{t}\n{im.size}')
    ax.axis('off')
plt.tight_layout()
plt.show()

print('💡 resize(): 지정한 크기로 변경 (비율 무관)')
print('💡 thumbnail(): 최대 크기 내에서 비율 유지 축소')
print('💡 crop((left, upper, right, lower)): 영역 잘라내기')
print('💡 rotate(각도, expand=True): 회전 시 캔버스 확장')

## 3. 색상 모드 변환

이미지의 색상 모드를 변환할 수 있습니다.

| 모드 | 설명 | 채널 |
|------|------|------|
| RGB | 컬러 (Red, Green, Blue) | 3 |
| RGBA | 컬러 + 투명도 | 4 |
| L | 흑백 (Luminance) | 1 |
| P | 팔레트 (256색) | 1 |
| CMYK | 인쇄용 컬러 | 4 |

In [ ]:
# ┌─────────────────────────────────────────┐
# │  색상 모드 변환                          │
# │  RGB <-> L <-> RGBA <-> P 모드 변환     │
# └─────────────────────────────────────────┘

# 컬러 이미지 생성
img = Image.new('RGB', (200, 200), (100, 150, 255))
draw = ImageDraw.Draw(img)
draw.rectangle([30, 30, 170, 170], fill=(255, 100, 50))
draw.ellipse([60, 60, 140, 140], fill=(0, 200, 100))

# 모드 변환
img_gray = img.convert('L')          # 흑백
img_rgba = img.convert('RGBA')        # RGBA
img_p = img.convert('P', palette=Image.Palette.ADAPTIVE)  # 팔레트

# 세피아 효과 (흑백 → 커스텀 컬러)
gray_arr = np.array(img_gray)
sepia_r = np.clip(gray_arr * 1.07, 0, 255).astype(np.uint8)
sepia_g = np.clip(gray_arr * 0.74, 0, 255).astype(np.uint8)
sepia_b = np.clip(gray_arr * 0.43, 0, 255).astype(np.uint8)
sepia_img = Image.fromarray(np.stack([sepia_r, sepia_g, sepia_b], axis=2))

# 모드별 픽셀 확인
print('=== 모드별 픽셀 값 ===')
print(f'RGB 픽셀 (50,50): {img.getpixel((50, 50))}')
print(f'L 픽셀 (50,50):   {img_gray.getpixel((50, 50))}')
print(f'RGBA 픽셀 (50,50): {img_rgba.getpixel((50, 50))}')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['RGB (원본)', 'L (흑백)', 'P (팔레트)', '세피아 효과']
images = [img, img_gray, img_p.convert('RGB'), sepia_img]
for ax, im, t in zip(axes, images, titles):
    ax.imshow(im)
    ax.set_title(t)
    ax.axis('off')
plt.tight_layout()
plt.show()

print('\n💡 convert("L"): 흑백 변환 (NTSC 가중치: 0.299R + 0.587G + 0.114B)')
print('💡 convert("P"): 256색 팔레트로 변환 (GIF 등에 사용)')
print('💡 convert("RGBA"): 투명도 채널 추가')

## 4. EXIF 메타데이터와 이미지 정보

JPEG 사진에는 촬영 정보(EXIF)가 포함되어 있습니다.
카메라 모델, 촬영일시, 노출, GPS 등의 정보를 읽을 수 있습니다.

In [ ]:
# ┌─────────────────────────────────────────┐
# │  EXIF 메타데이터와 이미지 정보             │
# │  _getexif(), 이미지 속성 확인              │
# └─────────────────────────────────────────┘

from PIL.ExifTags import TAGS

# 테스트 이미지
img = Image.new('RGB', (800, 600), (100, 180, 220))
draw = ImageDraw.Draw(img)
draw.rectangle([100, 100, 700, 500], fill=(240, 230, 210), outline='black', width=2)
draw.text((150, 200), 'Sample Photo 800x600', fill='black')

# 이미지 기본 정보
print('=== 이미지 기본 정보 ===')
print(f'크기: {img.size}          # (너비, 높이)')
print(f'모드: {img.mode}')
print(f'포맷: {img.format}')

# EXIF 태그 이름 확인
print('\n=== 주요 EXIF 태그 ===')
important_tags = {
    271: 'Make (제조사)',
    272: 'Model (모델)',
    306: 'DateTime (촬영일시)',
    33434: 'ExposureTime (노출시간)',
    36867: 'DateTimeOriginal (원본촬영일시)',
    37386: 'FocalLength (초점거리)',
}
for tag_id, name in important_tags.items():
    tag_name = TAGS.get(tag_id, f'Tag {tag_id}')
    print(f'  {tag_id:>6d}: {tag_name:<25s} -> {name}')

print('\n💡 실제 JPEG 파일에서 EXIF 읽기:')
print('  exif = img._getexif()')
print('  for tag_id, value in exif.items():')
print('      tag_name = TAGS.get(tag_id, tag_id)')
print("      print(f'{tag_name}: {value}')")

print('\n💡 EXIF Orientation 태그:')
print('  1: 보통 (회전 없음)')
print('  3: 180도 회전')
print('  6: 시계방향 90도 회전')
print('  8: 시계반대방향 90도 회전')
print('  -> ImageOps.exif_transpose(img)로 자동 보정 가능')

img

## 🎯 연습 문제

1. 300x200 흰 배경 이미지에 파란 원 3개를 그리고 PNG로 저장/불러오기를 수행하세요.
2. 이미지를 세 가지 방법(`resize`, `thumbnail`, `crop`)으로 축소하고 결과를 비교하세요.
3. RGB 이미지를 L(흑백)으로 변환한 후 다시 RGB로 변환하고, 원본과의 차이를 설명하세요.
4. `Image.new('RGBA', ...)`로 반투명 이미지를 만들고, `Image.alpha_composite()`로 합성하세요.
5. 실제 JPEG 파일의 EXIF 정보를 읽어 촬영일시와 카메라 모델을 출력하세요.